# Byte Pair Encoding (BPE): The Algorithm Behind LLM Tokenization

The algorithm:
  1. Start with a vocabulary of the individual characters in the text.
  2. Count adjacent pairs of symbols and choose the most frequent pair.
  3. Add the merged symbol to the vocabulary and replace every
     occurrence of that pair with it.
  4. Repeat steps 2 and 3 k times.

In [3]:
from __future__ import annotations
import argparse
import re
from collections import Counter

### Input Sentence and parameter, K:

In [6]:
TEXT = "The little cat sat on the mat with another cat."
K = 4

In [9]:
# Replace each adjacent occurrence of pair, e.g. ['c','a','t'] + ('a','t') -> ['c','at'].
def merge_pair(symbols, pair):
    result, i = [], 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            result.append(pair[0] + pair[1])
            i += 2
        else:
            result.append(symbols[i])
            i += 1
    return result


# Learn k merges from text. Returns (merges, vocabulary, tokens).
def train_bpe(text, k):
    
    words = re.findall(r"\w+|[^\w\s]", text.lower())       
    counts = Counter(words)

  
    corpus = {word: list(word) for word in counts}
    vocabulary = sorted(set("".join(words)))
    merges = []

    for _ in range(k):
        pairs = Counter()
        for word, symbols in corpus.items():
            for pair in zip(symbols, symbols[1:]):
                pairs[pair] += counts[word]
        if not pairs:
            break
        best = min(pairs, key=lambda p: (-pairs[p], p))      
        if pairs[best] < 2:                                
            break
      
        merges.append(best)
        vocabulary.append(best[0] + best[1])
        corpus = {word: merge_pair(symbols, best) for word, symbols in corpus.items()}

    tokens = [token for word in words for token in corpus[word]]
    return merges, vocabulary, tokens

merges, vocabulary, tokens = train_bpe(TEXT, K)
token_ids = [vocabulary.index(t) for t in tokens]            # text -> numbers
n_chars = len("".join(re.findall(r"\S", TEXT)))

print("Merges:    ", ", ".join(f"{a}+{b} -> {a + b}" for a, b in merges))
print("Vocabulary:", len(vocabulary), "symbols:", " ".join(vocabulary))
print("Tokens:    ", tokens)
print("Token ids: ", token_ids)
print(f"Result:     {n_chars} characters -> {len(tokens)} tokens")

Merges:     a+t -> at, t+h -> th, th+e -> the, c+at -> cat
Vocabulary: 18 symbols: . a c e h i l m n o r s t w at th the cat
Tokens:     ['the', 'l', 'i', 't', 't', 'l', 'e', 'cat', 's', 'at', 'o', 'n', 'the', 'm', 'at', 'w', 'i', 'th', 'a', 'n', 'o', 'the', 'r', 'cat', '.']
Token ids:  [16, 6, 5, 12, 12, 6, 3, 17, 11, 14, 9, 8, 16, 7, 14, 13, 5, 15, 1, 8, 9, 16, 10, 17, 0]
Result:     38 characters -> 25 tokens
